In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('../pca_full')  # Adjust the path as needed
from pca_full import pca_full

# Ensure reproducibility
np.random.seed(42)

# Set default text interpreter to LaTeX (for matplotlib)
plt.rcParams['text.usetex'] = True

# Define the OHspecial_transform function
def OHspecial_transform(df):
    names = df.columns.values
    data = df.values
    labels = []
    
    for j in range(len(names)):
        col_data = data[:, j]
        temp = np.unique(col_data[~np.isnan(col_data)])
        temp = temp[temp >= 0]
        
        if len(temp) == 0:
            # All data in column j is NaN or negative
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        elif len(temp) <= 2:
            # Keep original data
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        else:
            # One-hot encode
            var = np.zeros((data.shape[0], len(temp)))
            for l in range(len(temp)):
                var[:, l] = np.where(col_data == temp[l], 1, 0)
            idx = np.where(np.isnan(col_data))[0]
            var[idx, :] = np.nan
            # Use str instead of int to match MATLAB's num2str behavior
            label = [f"{names[j]}_{str(temp[l])}" for l in range(len(temp))]
        
        if j == 0:
            OH = var
        else:
            OH = np.hstack((OH, var))
        labels.extend(label)
    
    OH_df = pd.DataFrame(OH, columns=labels)
    return OH_df

# Read in data and remove non-cultural features + features missing > 50% of possible values

# From EA
KinOrgRaw = pd.read_csv('KinshipOrgDataRaw.csv')
SubRaw = pd.read_csv('SubsistenceDataRaw.csv')
# EA_All_Raw = pd.read_csv('EA_All.csv')  # Commented out in MATLAB code

# From Pulotu
IsoRaw = pd.read_csv('IsoDataRaw.csv')
ReligRaw = pd.read_csv('RelDataRaw.csv')

# Other datasets
Binford = pd.read_csv('Binford_all.csv')
Seshat = pd.read_csv('Seshat.csv')
Birds = pd.read_csv('data_birds_OH.csv')

# Apply OHspecial_transform to datasets
KinOrg = OHspecial_transform(KinOrgRaw)
Sub = OHspecial_transform(SubRaw)
Iso = OHspecial_transform(IsoRaw)
Relig = OHspecial_transform(ReligRaw)

for w in range(2, 3):

    if w == 1:
        X = KinOrg
    elif w == 2:
        X = Sub
    elif w == 3:
        X = Relig
    elif w == 4:
        X = Iso
    elif w == 5:
        X = Binford
    elif w == 6:
        X = Seshat
    elif w == 7:
        X = Birds

    # # Remove features missing > 50% of their entries
    dim = X.shape
    X = X.iloc[:, 0:dim[1]]  # Redundant but kept for similarity
    cols = X.columns.values
    X = X.to_numpy()
    counts = []
    for j in range(dim[1]):
        temp = np.isnan(X[:, j]).astype(int)
        idx = np.where(temp == 1)[0]
        counts.append(len(idx))

    counts = np.array(counts) / dim[0]
    idx = np.where(counts > 0.5)[0]
    cols = np.delete(cols, idx)
    X = np.delete(X, idx, axis=1)

    dim = X.shape

    # Compute initial mean and standard deviation, ignoring NaNs
    MnInit = np.tile(np.nanmean(X, axis=0), (dim[0], 1))
    STDInit = np.tile(np.nanstd(X, axis=0), (dim[0], 1))
    # pd.DataFrame(X_).to_csv("StdInit_full_data_dataset2.csv", index=False)

    if w < 5:
        X_ = X - MnInit
    elif w == 5:
        X_ = X
    elif w == 6:
        X_ = (X - MnInit) / STDInit
    elif w == 7:
        X_ = X

    X_ = X_.T

    MaxStop = min(dim)
    if MaxStop > 100:
        MaxStop = 100
    zz = np.arange(2, MaxStop + 1)

    # Initialize lists for dynamic data collection
    rms = []
    accu = []
    rmsA = []
    costA = []
    var_exp = []

    for y in range(MaxStop - 1):

        z = zz[y]
        print('###############################')
        print(f'Number of PCs: {z}')
        print('###############################')
        if y > 0:
            time.sleep(1)

        for k in range(1):  # Change if you want to do replications

            opts = {
                'maxiters': 80,
                'algorithm': 'vb',
                'uniquesv': 0,
                'rmsstop': [80, np.finfo(float).eps, np.finfo(float).eps],
                'cfstop': [80, 0, 0],
                'minangle': 0
            }

    
            
            # Print the scalar parameter z
            print(f"z = {repr(z)}")
            print()
            
            # Print the dictionary opts
            print("opts = {")
            for key, value in opts.items():
                print(f"    {repr(key)}: {repr(value)},")
            print("}")
            print()
            
            # Print the function call
            print("result = pca_full(X_, z, **opts)")

            result = pca_full(X_, z, **opts)
            A = result['A']
            S = result['S']
            Mu = result['Mu']
            V = result['V']
            cv = result['cv']
            hp = result['hp']
            lc = result['lc']
